In [ ]:
# ------------------------------------------------------------------------------
# Cell 1: Install Dependencies
# ------------------------------------------------------------------------------
!pip install fastapi uvicorn nest_asyncio requests huggingface_hub python-multipart
!pip install av comfyui-frontend-package comfyui-workflow-templates spandrel torchsde trampoline sentencepiece protobuf --no-deps

In [ ]:
# ------------------------------------------------------------------------------
# Cell 2: Imports & Configuration
# ------------------------------------------------------------------------------
import os
import sys
import time
import subprocess
import shutil
from pathlib import Path
import nest_asyncio

nest_asyncio.apply()

class CONFIG:
    COMFY_DIR = "/kaggle/working/ComfyUI"
    COMFY_PORT = 8000
    FACADE_PORT = 8001
    OUTPUTS_DIR = "/kaggle/working/comfy_outputs"
    ZROK_TOKEN_PATH = "/kaggle/input/sage-zrok-token/.zrok_api_key"
    ZROK_BINARY = "./zrok"
    ZROK_VERSION = "1.1.3"

Path(CONFIG.OUTPUTS_DIR).mkdir(parents=True, exist_ok=True)
Path(CONFIG.COMFY_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
# ------------------------------------------------------------------------------
# Cell 3: Setup FLUX 2 KLEIN (Text-to-Image)
# ------------------------------------------------------------------------------
import os
import subprocess
import shutil

COMFY_DIR = CONFIG.COMFY_DIR

# 1. Clone ComfyUI
if not os.path.exists(os.path.join(COMFY_DIR, "main.py")):
    print("Cloning ComfyUI...")
    if os.path.exists(COMFY_DIR): shutil.rmtree(COMFY_DIR)
    subprocess.run(f"git clone -q https://github.com/comfyanonymous/ComfyUI {COMFY_DIR}", shell=True, check=True)

# 2. Install Requirements
print("Installing ComfyUI Requirements...")
subprocess.run(f"pip install -r {COMFY_DIR}/requirements.txt", shell=True)

# 3. Directory Structure
dirs = {
    "text_encoders": f"{COMFY_DIR}/models/text_encoders",
    "diffusion": f"{COMFY_DIR}/models/diffusion_models",
    "vae": f"{COMFY_DIR}/models/vae"
}
for d in dirs.values(): os.makedirs(d, exist_ok=True)

# 4. Download Helper
def download_quiet(url, directory):
    filename = url.split('/')[-1]
    dest = os.path.join(directory, filename)
    if not os.path.exists(dest):
        print(f"⏳ Downloading {filename}...")
        subprocess.run(f"wget -nv -c -O '{dest}' '{url}'", shell=True, check=True)
        print(f"✅ Downloaded {filename}")
    else:
        print(f"✅ {filename} already exists.")

print("Downloading Flux 2 Klein T2I Weights... (Logs will be minimal)")

downloads = [
    # Qwen 3.4B Text Encoder
    (dirs["text_encoders"], "https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors"),
    
    # Flux 2 Klein Standard (T2I) - This is the SMART instruction-following model
    (dirs["diffusion"], "https://huggingface.co/black-forest-labs/FLUX.2-klein-4b-fp8/resolve/main/flux-2-klein-4b-fp8.safetensors"),
    
    # Flux 2 VAE
    (dirs["vae"], "https://huggingface.co/Comfy-Org/flux2-dev/resolve/main/split_files/vae/flux2-vae.safetensors")
]

for directory, url in downloads:
    download_quiet(url, directory)

print("✅ All Flux 2 Klein T2I models ready.")

In [ ]:
# ------------------------------------------------------------------------------
# Cell 4: Additional Setup
# ------------------------------------------------------------------------------
import os
from pathlib import Path

# Ensure Input folder exists (though not used for T2I, good practice)
Path("/kaggle/working/ComfyUI/input").mkdir(parents=True, exist_ok=True)

In [ ]:
# ------------------------------------------------------------------------------
# Cell 5: Create Headless API Script (flux_api_t2i.py)
# ------------------------------------------------------------------------------
# Contains the Negative Prompt Logic and 900s Timeout

code_content = """
import os
import sys
import json
import time
import uuid
import requests
import uvicorn
from pathlib import Path
from typing import Optional
from fastapi import FastAPI, Form, File, UploadFile, BackgroundTasks
from fastapi.responses import StreamingResponse, JSONResponse

# --- CONFIG ---
class FLUX_CONFIG:
    COMFY_PORT = 8000
    FACADE_PORT = 8001
    OUTPUTS_DIR = "/kaggle/working/comfy_outputs"
    
    # Models (T2I Specific)
    UNET_NAME = "flux-2-klein-4b-fp8.safetensors"
    CLIP_NAME = "qwen_3_4b.safetensors"
    VAE_NAME = "flux2-vae.safetensors"

# Ensure directories
Path(FLUX_CONFIG.OUTPUTS_DIR).mkdir(parents=True, exist_ok=True)

# --- COMFY HELPERS ---
def comfy_post(path, json_payload=None): 
    try: return requests.post(f"http://localhost:{FLUX_CONFIG.COMFY_PORT}{path}", json=json_payload)
    except: return None

def comfy_get(path): 
    try: return requests.get(f"http://localhost:{FLUX_CONFIG.COMFY_PORT}{path}")
    except: return None

# --- WORKFLOW BUILDER (TXT2IMG) ---
def build_t2i_workflow(prompt, negative_prompt="", width=1024, height=1024, steps=20, seed=None):
    if seed is None: seed = int(uuid.uuid4().int & (2**31-1))
    
    wf = {
        "10": { "class_type": "UNETLoader", "inputs": { "unet_name": FLUX_CONFIG.UNET_NAME, "weight_dtype": "default" } },
        # FIX: Type 'flux2' is required
        "11": { "class_type": "CLIPLoader", "inputs": { "clip_name": FLUX_CONFIG.CLIP_NAME, "type": "flux2" } },
        "12": { "class_type": "VAELoader", "inputs": { "vae_name": FLUX_CONFIG.VAE_NAME } },
        
        # Empty Latent for T2I
        "3": { "class_type": "EmptyLatentImage", "inputs": { "width": width, "height": height, "batch_size": 1 } },
        
        # Positive Prompt
        "6":  { "class_type": "CLIPTextEncode", "inputs": { "text": prompt, "clip": ["11", 0] } },
        
        # Negative Prompt (FIXED)
        "7":  { "class_type": "CLIPTextEncode", "inputs": { "text": negative_prompt, "clip": ["11", 0] } },
        
        # Sampler
        "4":  { "class_type": "KSampler", "inputs": { 
            "model": ["10", 0], "positive": ["6", 0], "negative": ["7", 0], 
            "latent_image": ["3", 0], 
            "seed": seed, "steps": steps, "cfg": 1.0, "sampler_name": "euler", "scheduler": "simple", 
            "denoise": 1.0
        } },
        
        # Decode & Save
        "8":  { "class_type": "VAEDecode", "inputs": { "samples": ["4", 0], "vae": ["12", 0] } },
        "9":  { "class_type": "SaveImage", "inputs": { "filename_prefix": "flux_t2i", "images": ["8", 0] } }
    }
    return wf

def process_workflow_bg(workflow_dict, task_id):
    status_file = Path(FLUX_CONFIG.OUTPUTS_DIR) / f"task_{task_id}.json"
    status_file.write_text(json.dumps({"status": "processing"}))

    resp = comfy_post("/prompt", {"prompt": workflow_dict})
    if not resp:
        status_file.write_text(json.dumps({"status": "error", "detail": "ComfyUI unreachable"}))
        return

    try: prompt_id = resp.json().get("prompt_id")
    except: 
        status_file.write_text(json.dumps({"status": "error", "detail": "Invalid Comfy response"}))
        return

    # Poll Loop (900s timeout = 15 minutes)
    for _ in range(900):
        hist = comfy_get(f"/history/{prompt_id}")
        if hist and prompt_id in hist.json():
            data = hist.json()[prompt_id]
            if "status" in data and data["status"].get("status_str") == "error":
                 status_file.write_text(json.dumps({"status": "error", "detail": "ComfyUI Workflow Error"}))
                 return

            files = []
            for out in data.get("outputs", {}).values():
                for img in out.get("images", []):
                    fname = img["filename"]
                    raw = comfy_get(f"/view?filename={fname}&type=output")
                    local_p = Path(FLUX_CONFIG.OUTPUTS_DIR) / f"{prompt_id}_{fname}"
                    local_p.write_bytes(raw.content)
                    files.append(local_p.name)
            
            status_file.write_text(json.dumps({"status": "completed", "files": files}))
            return
        time.sleep(1)
    
    status_file.write_text(json.dumps({"status": "error", "detail": "Timeout"}))

# --- API DEFINITION ---
app = FastAPI(title="Flux 2 Klein T2I Client API")

@app.get("/")
def health():
    return {"status": "Flux 2 T2I API is Alive"}

@app.post("/generate-async")
async def generate_async(
    background_tasks: BackgroundTasks,
    prompt: str = Form(...),
    negative_prompt: str = Form(""),
    num_inference_steps: int = Form(20),
    height: int = Form(1024),
    width: int = Form(1024),
    guidance_scale: float = Form(None),
    seed: int = Form(None)
):
    task_id = uuid.uuid4().hex
    wf = build_t2i_workflow(prompt, negative_prompt, width, height, num_inference_steps, seed)
    background_tasks.add_task(process_workflow_bg, wf, task_id)
    return {"task_id": task_id, "status": "queued"}

@app.get("/status/{task_id}")
def get_status(task_id: str):
    p = Path(FLUX_CONFIG.OUTPUTS_DIR) / f"task_{task_id}.json"
    if not p.exists(): return {"status": "pending"}
    data = json.loads(p.read_text())
    if data.get("status") == "completed" and data.get("files"):
        img_path = Path(FLUX_CONFIG.OUTPUTS_DIR) / data["files"][0]
        if img_path.exists():
            return StreamingResponse(open(img_path, "rb"), media_type="image/png")
    return data

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=FLUX_CONFIG.FACADE_PORT)
"""

with open("flux_api_t2i.py", "w") as f:
    f.write(code_content)

print("✅ 'flux_api_t2i.py' created.")

In [ ]:
# ------------------------------------------------------------------------------
# Cell 6: Start System (Robust Zrok)
# ------------------------------------------------------------------------------
import subprocess
import sys
import time
import os
import re
import requests

# Config
COMFY_DIR = "/kaggle/working/ComfyUI"
COMFY_PORT = 8000
FACADE_PORT = 8001
ZROK_BINARY = "./zrok"
ZROK_TOKEN_PATH = "/kaggle/input/sage-zrok-token/.zrok_api_key"

def get_zrok_token():
    if os.path.isfile(ZROK_TOKEN_PATH): 
        with open(ZROK_TOKEN_PATH, "r") as f: return f.read().strip()
    return None

def force_zrok_auth(token):
    if not os.path.exists(ZROK_BINARY):
        print("⬇️ Downloading Zrok...")
        subprocess.run("wget -q https://github.com/openziti/zrok/releases/download/v1.1.3/zrok_1.1.3_linux_amd64.tar.gz -O zrok.tar.gz", shell=True)
        subprocess.run("tar -xzf zrok.tar.gz && chmod +x zrok", shell=True)
    subprocess.run("chmod +x ./zrok", shell=True)
    subprocess.run(f"{ZROK_BINARY} disable", shell=True, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    print("🔄 Authenticating Zrok...")
    res = subprocess.run(f'{ZROK_BINARY} enable --headless "{token}"', shell=True, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"❌ Zrok Enable Failed:\n{res.stderr}")
        return False
    print("✅ Zrok Authenticated.")
    return True

def start_zrok_tunnel_loop(port):
    print(f"   Opening tunnel for port {port}...")
    share_proc = subprocess.Popen([ZROK_BINARY, "share", "public", f"localhost:{port}", "--headless"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    for i in range(10):
        time.sleep(3)
        if share_proc.poll() is not None: return "Error"
        res = subprocess.run([ZROK_BINARY, "overview"], capture_output=True, text=True)
        m = re.search(r'(https?://[a-z0-9\-\.]+\.zrok\.io)', res.stdout)
        if m: return m.group(1)
    return "Timeout"

# 1. CLEANUP
print("🧹 Killing old processes...")
os.system(f"fuser -k {COMFY_PORT}/tcp")
os.system(f"fuser -k {FACADE_PORT}/tcp")
os.system("pkill -9 zrok")
os.system("pkill -f flux_api_t2i.py")
time.sleep(2)

# 2. START COMFY
print("🚀 Starting ComfyUI Server...")
comfy_proc = subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)], cwd=COMFY_DIR, stdout=subprocess.DEVNULL)

print("⏳ Waiting for ComfyUI...")
comfy_ready = False
for i in range(60):
    try:
        if requests.get(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=1).status_code == 200:
            print("✅ ComfyUI is Ready.")
            comfy_ready = True
            break
    except: pass
    time.sleep(1)

if comfy_ready:
    # 3. START HEADLESS API
    print(f"🚀 Starting Headless API (flux_api_t2i.py) on Port {FACADE_PORT}...")
    api_proc = subprocess.Popen([sys.executable, "flux_api_t2i.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    time.sleep(2)
    
    # 4. START ZROK
    token = get_zrok_token()
    if token and force_zrok_auth(token):
        url = start_zrok_tunnel_loop(FACADE_PORT)
        print(f"\n========================================================")
        print(f"🌍 T2I API URL: {url}")
        print(f"   Generate: POST {url}/generate-async")
        print(f"========================================================\n")
        try:
            while True:
                if api_proc.poll() is not None:
                    print("❌ API Died! Log:")
                    print(api_proc.stderr.read().decode())
                    break
                time.sleep(60)
        except KeyboardInterrupt:
            comfy_proc.terminate()
            api_proc.terminate()
            os.system("pkill zrok")
    else:
        print("⚠️ Zrok Token invalid or missing.")
else:
    print("❌ ComfyUI Failed to Start.")